## Neo4J setup

### Task 1 Refactor data


In [9]:
import sys
sys.path.insert(0, "..")

from src.data_prep import load_and_prepare

df = load_and_prepare("../data/HI-Small_Trans.csv")

print("Shape:", df.shape)
print("Timestamp dtype:", df["Timestamp"].dtype)
print("Chronological:", df["Timestamp"].is_monotonic_increasing)

print(
    df[
        ["From Bank", "Account", "sender_id",
         "To Bank", "Account.1", "receiver_id"]
    ].head()
)

Shape: (5078345, 13)
Timestamp dtype: datetime64[us]
Chronological: True
   From Bank    Account        sender_id  To Bank  Account.1      receiver_id
0        121  8123FB9B0    121_8123FB9B0      121  8123FB9B0    121_8123FB9B0
1      25170  8095AF7C0  25170_8095AF7C0    25170  8095AF7C0  25170_8095AF7C0
2      25665  809A7D4B0  25665_809A7D4B0    24779  809189BA0  24779_809189BA0
3      32317  800D4E490  32317_800D4E490    12004  800D4E750  12004_800D4E750
4       1024  800C8D9D0   1024_800C8D9D0     1024  800C8D9D0   1024_800C8D9D0


### Task 2: Design Neo4j Graph


##### Concepts to understand:
- Node
- Relationship
- Property
- Label
- MERGE
- CREATE
- Constraint

### Neo4j Graph Schema

### Nodes
`Account`

Identity:
`Bank ID + Account ID`

Properties:
- `id`
- `bank_id`
- `account_id`

### Relationships
`TRANSFER`

Direction:
`sender -> receiver`

Properties:
- `timestamp`
- `amount_paid`
- `amount_received`
- `payment_currency`
- `receiving_currency`
- `payment_format`
- `is_laundering`

#### Executed in Neo4j browser

- CREATE CONSTRAINT account_id_unique IF NOT EXISTS
FOR (a:Account) 
REQUIRE a.id IS UNIQUE;

- SHOW CONTRAINTS;

### TASK 3: Load 10 transaction in neo4j

Concepts to understand:

- Neo4j Python driver
- GraphDatabase.driver
- verify_connectivity()
- execute_query()
- query parameters

In [10]:
from neo4j import GraphDatabase
import os
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

def get_driver():
    password = os.environ("NEO4J_PASSWORD")

    driver = GraphDatabase.driver(
        URI,
        auth = (USERNAME, password)
    )
    driver.verify_connectivity

    return driver


In [11]:
def load_transaction(driver, row):
    query = """
    MERGE (sender:Account {id: $sender_id})
    SET sender.bank_id = $sender_bank,
        sender.account_id = $sender_account

    MERGE (receiver:Account {id: $receiver_id})
    SET receiver.bank_id = $receiver_bank,
        receiver.account_id = $receiver_account

    CREATE (sender)-[t:TRANSFER {
        timestamp: $timestamp,
        amount_paid: $amount_paid,
        amount_received: $amount_received,
        payment_currency: $payment_currency,
        receiving_currency: $receiving_currency,
        payment_format: $payment_format,
        is_laundering: $is_laundering
    }]->(receiver)

    RETURN sender.id AS sender, receiver.id AS receiver
    """

    return driver.execute_query(
        query,
        sender_id=str(row["sender_id"]),
        sender_bank=int(row["From Bank"]),
        sender_account=str(row["Account"]),
        receiver_id=str(row["receiver_id"]),
        receiver_bank=int(row["To Bank"]),
        receiver_account=str(row["Account.1"]),
        timestamp=row["Timestamp"].to_pydatetime(),
        amount_paid=float(row["Amount Paid"]),
        amount_received=float(row["Amount Received"]),
        payment_currency=str(row["Payment Currency"]),
        receiving_currency=str(row["Receiving Currency"]),
        payment_format=str(row["Payment Format"]),
        is_laundering=int(row["Is Laundering"]),
        database_=DATABASE,
    )

In [12]:
import sys
sys.path.insert(0, "..")
from src.neo4j_client import get_driver
from src.data_prep import load_transactions
driver = get_driver()

sample = df.head(10)

for _, row in sample.iterrows():
    load_transaction(driver, row)

print("10 transactions loaded")

10 transactions loaded


### Task 4: Validate before full import

My reference for CYPHER:
https://neo4j.com/docs/cypher-manual/current/cheat-sheet/

##Run in NEO4J.  
  
##### Relationship count
- MATCH ()-[t:TRANSFER]->()     
RETURN count(t) AS transactions;
expected 10

##### Account count
-  MATCH (a:Account)    
RETURN count(a) AS accounts;    

##### Inspect the graph
- MATCH (a:Account)-[t:TRANSFER]->(b:Account)   
RETURN a, t, b  
LIMIT 10;   

##### Inspect transaction properties 
- MATCH (a:Account)-[t:TRANSFER]->(b:Account)   
RETURN  
    a.id AS sender, 
    b.id AS receiver,   
    t.timestamp AS timestamp,   
    t.amount_paid AS amount_paid,   
    t.amount_received AS amount_received,   
    t.payment_format AS payment_format,     
    t.is_laundering AS laundering   
LIMIT 10;   

##### Validate repeated transactions

- MATCH (a:Account)-[t:TRANSFER]->(b:Account)   
RETURN a.id, b.id, count(t) AS transaction_count    
ORDER BY transaction_count DESC;    



## 2. Full Neo4j Import Preparation

In [13]:
from pathlib import Path
import pandas as pd

import_dir = Path("../data/processed/neo4j_import")
import_dir.mkdir(parents=True, exist_ok=True)

accounts_file = import_dir / "accounts.csv"
transactions_file = import_dir / "transactions.csv"

### Create Accounts CSV

In [14]:
senders = df[
    ["sender_id", "From Bank", "Account"]
].rename(
    columns={
        "sender_id": "id",
        "From Bank": "bank_id",
        "Account": "account_id"
    }
)

receivers = df[
    ["receiver_id", "To Bank", "Account.1"]
].rename(
    columns={
        "receiver_id": "id",
        "To Bank": "bank_id",
        "Account.1": "account_id"
    }
)

accounts = (
    pd.concat([senders, receivers], ignore_index=True)
    .drop_duplicates(subset="id")
)

# Neo4j bulk-import headers
accounts = accounts.rename(columns={"id": "id:ID(Account)"})
accounts[":LABEL"] = "Account"

accounts.to_csv(
    accounts_file,
    index=False
)

print(f"Unique accounts: {len(accounts):,}")
print(accounts.head())

Unique accounts: 515,088
    id:ID(Account)  bank_id account_id   :LABEL
0    121_8123FB9B0      121  8123FB9B0  Account
1  25170_8095AF7C0    25170  8095AF7C0  Account
2  25665_809A7D4B0    25665  809A7D4B0  Account
3  32317_800D4E490    32317  800D4E490  Account
4   1024_800C8D9D0     1024  800C8D9D0  Account


### Create tranasaction CSV

In [15]:
transactions = df[
    [
        "sender_id",
        "receiver_id",
        "Timestamp",
        "Amount Paid",
        "Amount Received",
        "Payment Currency",
        "Receiving Currency",
        "Payment Format",
        "Is Laundering"
    ]
].copy()

transactions = transactions.rename(
    columns={
        "sender_id": ":START_ID(Account)",
        "receiver_id": ":END_ID(Account)",
        "Timestamp": "timestamp",
        "Amount Paid": "amount_paid",
        "Amount Received": "amount_received",
        "Payment Currency": "payment_currency",
        "Receiving Currency": "receiving_currency",
        "Payment Format": "payment_format",
        "Is Laundering": "is_laundering"
    }
)

transactions["timestamp"] = (
    pd.to_datetime(transactions["timestamp"])
    .dt.strftime("%Y-%m-%dT%H:%M:%S")
)

transactions[":TYPE"] = "TRANSFER"

transactions.to_csv(
    transactions_file,
    index=False,
    header=[
        ":START_ID(Account)",
        ":END_ID(Account)",
        "timestamp:localdatetime",
        "amount_paid:double",
        "amount_received:double",
        "payment_currency:string",
        "receiving_currency:string",
        "payment_format:string",
        "is_laundering:int",
        ":TYPE"
    ]
)

print(f"Saved: {transactions_file}")

Saved: ../data/processed/neo4j_import/transactions.csv


### Checking files before sending them to neo4j

In [16]:
print(pd.read_csv(transactions_file).head())

  :START_ID(Account) :END_ID(Account) timestamp:localdatetime  \
0      121_8123FB9B0    121_8123FB9B0     2022-09-01T00:00:00   
1    25170_8095AF7C0  25170_8095AF7C0     2022-09-01T00:00:00   
2    25665_809A7D4B0  24779_809189BA0     2022-09-01T00:00:00   
3    32317_800D4E490  12004_800D4E750     2022-09-01T00:00:00   
4     1024_800C8D9D0   1024_800C8D9D0     2022-09-01T00:00:00   

   amount_paid:double  amount_received:double payment_currency:string  \
0               47.64                   47.64             Saudi Riyal   
1             3917.42                 3917.42         Canadian Dollar   
2               97.49                   97.49         Canadian Dollar   
3            13939.05                13939.05                    Euro   
4               10.37                   10.37                    Euro   

  receiving_currency:string payment_format:string  is_laundering:int     :TYPE  
0               Saudi Riyal          Reinvestment                  0  TRANSFER  
1       

In [17]:
print(df["Is Laundering"].value_counts())

Is Laundering
0    5073168
1       5177
Name: count, dtype: int64


In [18]:
print(accounts.head())
print(transactions.head())

print("\nAccounts:", len(accounts))
print("Transactions:", len(transactions))

print("\nFiles:")
print(accounts_file)
print(transactions_file)

    id:ID(Account)  bank_id account_id   :LABEL
0    121_8123FB9B0      121  8123FB9B0  Account
1  25170_8095AF7C0    25170  8095AF7C0  Account
2  25665_809A7D4B0    25665  809A7D4B0  Account
3  32317_800D4E490    32317  800D4E490  Account
4   1024_800C8D9D0     1024  800C8D9D0  Account
  :START_ID(Account) :END_ID(Account)            timestamp  amount_paid  \
0      121_8123FB9B0    121_8123FB9B0  2022-09-01T00:00:00        47.64   
1    25170_8095AF7C0  25170_8095AF7C0  2022-09-01T00:00:00      3917.42   
2    25665_809A7D4B0  24779_809189BA0  2022-09-01T00:00:00        97.49   
3    32317_800D4E490  12004_800D4E750  2022-09-01T00:00:00     13939.05   
4     1024_800C8D9D0   1024_800C8D9D0  2022-09-01T00:00:00        10.37   

   amount_received payment_currency receiving_currency payment_format  \
0            47.64      Saudi Riyal        Saudi Riyal   Reinvestment   
1          3917.42  Canadian Dollar    Canadian Dollar   Reinvestment   
2            97.49  Canadian Dollar    Can

In [ ]:
print(transactions[":TYPE"].value_counts())

:TYPE
TRANSFER    5078345
Name: count, dtype: int64


Next steps:     
run in terminal - 
brew services stop neo4j 
neo4j status            
- neo4j-admin database import full neo4j \
  --overwrite-destination=true \
  --nodes=data/processed/neo4j_import/accounts.csv \
  --relationships=data/processed/neo4j_import/transactions.csv


2) Start neo4j again
- MATCH (a:Account)
RETURN count(a) AS accounts;
- MATCH ()-[t:TRANSFER]->()
RETURN t.is_laundering AS laundering,
       count(*) AS count
ORDER BY laundering;
- MATCH ()-[t:TRANSFER]->()
WHERE t.is_laundering = 1
RETURN count(t) AS laundering_transactions;

validate the numbers

